In [2]:
import os
import pandas as pd
import numpy as np

# Project root = folder ABOVE this notebook
BASE_DIR = os.path.dirname(os.getcwd())

# Folder with 150 CLEAN intraday files
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "clean_intraday")

# Output file for block-level liquidity summary
blocks_path = os.path.join(BASE_DIR, "Indian_microstructure","data", "processed", "intraday_block_liquidity.csv")

print("Project root:", BASE_DIR)
print("Cleaned dir exists:", os.path.exists(PROCESSED_DIR))
print("Number of cleaned files:", len(os.listdir(PROCESSED_DIR)))
print("Block summary file:", blocks_path)
print("Block file exists already:", os.path.exists(blocks_path))


Project root: C:\
Cleaned dir exists: True
Number of cleaned files: 150
Block summary file: C:\Indian_microstructure\data\processed\intraday_block_liquidity.csv
Block file exists already: False


In [3]:
def summarize_one_stock_blocks(clean_path: str) -> pd.DataFrame:
    """
    Given a CLEAN intraday CSV (one stock),
    return a per-day, per-block summary with dead_fraction.
    Blocks:
      - OPEN   : 09:15–10:00
      - MIDDAY : 10:00–13:00
      - CLOSE  : 13:00–15:30
    """
    df = pd.read_csv(clean_path, parse_dates=["timestamp"])
    
    filename = os.path.basename(clean_path)
    symbol = filename.split("__")[0]
    
    # Date for grouping
    df["date"] = df["timestamp"].dt.date

    # Minutes since midnight (vectorized)
    mins = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute

    # Time block boundaries in minutes
    open_start  = 9 * 60 + 15   # 09:15
    open_end    = 10 * 60       # 10:00
    mid_start   = 10 * 60       # 10:00
    mid_end     = 13 * 60       # 13:00
    # Rest until 15:30 is CLOSE

    conds = [
        (mins >= open_start) & (mins < open_end),
        (mins >= mid_start)  & (mins < mid_end)
    ]
    choices = ["OPEN", "MIDDAY"]
    df["block"] = np.select(conds, choices, default="CLOSE")

    # Group by date + block
    summary = (
        df.groupby(["date", "block"])
          .agg(
              total_minutes=("timestamp", "count"),
              traded_minutes=("volume", lambda x: (x > 0).sum()),
              zero_minutes=("volume",  lambda x: (x == 0).sum())
          )
          .reset_index()
    )

    summary["dead_fraction"] = summary["zero_minutes"] / summary["total_minutes"]
    summary["symbol"] = symbol

    summary = summary[["symbol", "date", "block",
                       "total_minutes", "traded_minutes",
                       "zero_minutes", "dead_fraction"]]
    return summary


In [12]:
pattern = os.path.join(RAW_DIR, "*.csv")
all_files = glob.glob(pattern)

print("Total CSV files found:", len(all_files))

equity_files =[]
index_files =[]
master_file = None

for path in all_files:
    name = os.path.basename(path)
    name_lower = name.lower()

    if name_lower =="master.csv":
        master_file = path
    elif "__INDICES__" in name.upper():
        index_files.append(path)
    else:
        equity_files.append(path)
print("Equity files:" , len(equity_files))
print("Index files:", len(index_files))
print("Master file:", master_file)

Total CSV files found: 160
Equity files: 150
Index files: 9
Master file: C:\Indian_microstructure\data\raw\intraday_\FullDataCsv\master.csv


In [13]:
import pandas as pd

row_counts =[]

for path in equity_files:
    df = pd.read_csv(path, usecols=["timestamp"])
    row_counts.append(len(df))

print("Number of equity symbols:", len(equity_files))
print("Smallest file(rows:" , min(row_counts))
print("Largest file(rows):" , max(row_counts))

sample_path= equity_files[0]
print("Sample equity files:", os.path.basename(sample_path))
sample_df = pd.read_csv(sample_path)
print("Sample shape:", sample_df.shape)
sample_df.head()

Number of equity symbols: 150
Smallest file(rows: 114467
Largest file(rows): 370546
Sample equity files: AARTIIND__EQ__NSE__NSE__MINUTE.csv
Sample shape: (370458, 6)


,timestamp,open,high,low,close,volume
0,2017-01-02 09:15:00+05:30,340.0,340.0,340.0,340.0,11.0
1,2017-01-02 09:16:00+05:30,340.0,340.0,340.0,340.0,0.0
2,2017-01-02 09:17:00+05:30,340.0,340.0,340.0,340.0,0.0
3,2017-01-02 09:18:00+05:30,340.0,343.7,340.0,343.7,1.0
4,2017-01-02 09:19:00+05:30,343.7,343.7,343.7,343.7,1.0


In [14]:
import pandas as pd
missing_timestamp =[]

for path in equity_files:
    cols =pd.read_csv(path, nrows=0).columns
    cols_lower =[c.lower() for c in cols]

    if "timestamp" not in cols_lower:
        missing_timestamp.append((os.path.basename(path), list(cols)))
len(missing_timestamp), missing_timestamp[:5]

(0, [])

In [5]:
import pandas as pd
import os

baj_path = None
for path in equity_files:
    if "BAJFINANCE" in os.path.basename(path):
        baj_path =path
        break
print("BAJFINANCE file:", os.path.basename(baj_path))
df = pd.read_csv(baj_path)

df["timestamp"] = pd.to_datetime(df["timestamp"])

first_day = df["timestamp"].dt.date.min()
one_day_df = df[df["timestamp"].dt.date == first_day]

print("Trading day checked:", first_day)
print("Rows for this day:", len(one_day_df))

zero_volume = (one_day_df["volume"] == 0).sum()
null_volume = one_day_df["volume"].isna().sum()

print("zero-vol mins:",zero_volume)
print("Null-vol mins:", null_volume)

one_day_df.head()

BAJFINANCE file: BAJFINANCE__EQ__NSE__NSE__MINUTE.csv
Trading day checked: 2017-01-02
Rows for this day: 375
zero-vol mins: 0
Null-vol mins: 0


,timestamp,open,high,low,close,volume
0,2017-01-02 09:15:00+05:30,851.65,857.75,849.30,855.00,9208.0
1,2017-01-02 09:16:00+05:30,855.00,856.10,853.80,854.05,5092.0
2,2017-01-02 09:17:00+05:30,854.05,854.40,851.55,852.00,16448.0
3,2017-01-02 09:18:00+05:30,851.45,852.00,849.75,850.75,6170.0
4,2017-01-02 09:19:00+05:30,850.50,850.50,846.20,846.25,5669.0


In [6]:
import pandas as pd
import os

# 1. Find the AARTIIND file path
aarti_path = None
for path in equity_files:
    if "AARTIIND" in os.path.basename(path):
        aarti_path = path
        break

print("AARTIIND file:", os.path.basename(aarti_path))
“Loop finished, I see ~150 CLEAN csvs...
# 2. Load the full AARTIIND CSV
df = pd.read_csv(aarti_path)

# 3. Convert timestamp column into proper datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 4. Extract the first full trading day in the dataset
first_day = df["timestamp"].dt.date.min()
one_day_df = df[df["timestamp"].dt.date == first_day]

print("Trading day checked:", first_day)
print("Rows for this day:", len(one_day_df))

# 5. Volume diagnostics
zero_volume = (one_day_df["volume"] == 0).sum()
null_volume = one_day_df["volume"].isna().sum()

print("Zero-volume minutes:", zero_volume)
print("Null-volume minutes:", null_volume)

# 6. Quick visual sanity check
one_day_df.head()


SyntaxError: invalid character '“' (U+201C) (854787701.py, line 12)

In [ ]:
# Filter only the zero-volume minutes
zero_vol_df = one_day_df[one_day_df["volume"] == 0]

# Check if price ever changes during zero-volume periods
price_changes_during_zero_vol = (zero_vol_df["close"].diff().abs() > 0).sum()

print("Zero-volume minutes:", len(zero_vol_df))
print("Minutes where price changed during zero-volume:", price_changes_during_zero_vol)


In [ ]:
# For each zero-volume minute:
# Check whether its close equals the last traded close before it

# Create a column that holds the last traded close at ALL times
corrected_day["last_traded_close"] = corrected_day["close"].where(
    corrected_day["volume"] > 0
).ffill()

# Now compare for zero-volume rows
mismatch = (
    corrected_day.loc[zero_mask, "close"] 
    != corrected_day.loc[zero_mask, "last_traded_close"]
)

print("Zero-volume minutes:", zero_mask.sum())
print("Zero-volume minutes where close != last traded close:", mismatch.sum())


In [ ]:
import pandas as pd
import os

def clean_one_stock_file(path: str) -> pd.DataFrame:
    """
    Clean a single intraday CSV (one stock) so that:
      - timestamps are parsed and sorted
      - for zero-volume minutes, price is flat at last traded close
    Returns a cleaned DataFrame with the same rows but corrected OHLC.
    """

    # 1. Load raw CSV
    df = pd.read_csv(path)

    # 2. Parse timestamp to datetime and sort
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    # 3. Identify traded vs non-traded minutes
    trade_mask = df["volume"] > 0          # True when there was a trade
    zero_mask  = df["volume"] == 0         # True when there was no trade

    # 4. Build a "trade-only close" that is NaN when there was no trade
    df["close_trade_only"] = df["close"].where(trade_mask, pd.NA)

    # 5. Forward-fill last traded close over time
    df["last_traded_close"] = df["close_trade_only"].ffill()

    # 6. Use this as the final close (trade-driven price process)
    df["close"] = df["last_traded_close"]

    # 7. For zero-volume minutes, flatten the full bar to that close
    df.loc[zero_mask, "open"] = df.loc[zero_mask, "close"]
    df.loc[zero_mask, "high"] = df.loc[zero_mask, "close"]
    df.loc[zero_mask, "low"]  = df.loc[zero_mask, "close"]

    # 8. Optional integrity check: for zero-volume minutes, close must equal last_traded_close
    mismatches = (
        df.loc[zero_mask, "close"] != df.loc[zero_mask, "last_traded_close"]
    ).sum()
    if mismatches > 0:
        print(f"WARNING: {os.path.basename(path)} has {mismatches} mismatched zero-volume bars.")

    return df


In [ ]:
# BASE_DIR was defined earlier in your census notebook.
# If not, recreate it:
# import os
# BASE_DIR = os.path.dirname(os.getcwd())   # project root (one level above notebooks)

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "clean_intraday")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Processed data folder:", PROCESSED_DIR)


In [ ]:
from tqdm import tqdm  # optional progress bar; pip install tqdm if needed

# If tqdm isn't installed, you can comment it out and just do: for path in equity_files:
try:
    iterator = tqdm(equity_files, desc="Cleaning equities")
except:
    iterator = equity_files

for path in iterator:
    filename = os.path.basename(path)
    # Extract a shorter symbol name (before the first "__")
    symbol = filename.split("__")[0]

    # 1. Clean this stock's intraday data
    df_clean = clean_one_stock_file(path)

    # 2. Build output path (we'll keep a similar naming convention)
    out_name = f"{symbol}__CLEAN__NSE__MINUTE.csv"
    out_path = os.path.join(PROCESSED_DIR, out_name)

    # 3. Save cleaned data as CSV (raw data in data/raw stays untouched)
    df_clean.to_csv(out_path, index=False)


In [ ]:
pip install tqdm

In [ ]:
clean_files = os.listdir(PROCESSED_DIR)
len(clean_files), clean_files[:5]


In [ ]:
import pandas as pd
import os

# 1. Load ONE cleaned stock file: BAJFINANCE
baj_path = [p for p in os.listdir(PROCESSED_DIR) if p.startswith("BAJFINANCE")][0]
baj_path = os.path.join(PROCESSED_DIR, baj_path)

baj = pd.read_csv(baj_path, parse_dates=["timestamp"])

# 2. Add a DATE column (remove time)
baj["date"] = baj["timestamp"].dt.date

# 3. Now build the DAILY summary
daily_summary = (
    baj
    .groupby("date")
    .agg(
        total_minutes=("timestamp", "count"),
        traded_minutes=("volume", lambda x: (x > 0).sum()),
        zero_minutes=("volume", lambda x: (x == 0).sum())
    )
    .reset_index()
)

# 4. Add 'dead_fraction' = how dead the stock was that day
daily_summary["dead_fraction"] = (
    daily_summary["zero_minutes"] / daily_summary["total_minutes"]
)

daily_summary.head()


In [ ]:
import pandas as pd
import os

aarti_path = [p for p in os.listdir(PROCESSED_DIR) if p.startswith("AARTIIND")][0]
aarti_path = os.path.join(PROCESSED_DIR, aarti_path)

aarti = pd.read_csv(aarti_path, parse_dates=["timestamp"])
aarti.head()


In [ ]:
# Extract time from timestamp
aarti["time"] = aarti["timestamp"].dt.time
aarti["date"] = aarti["timestamp"].dt.date

# Define time blocks
def time_block(t):
    if t >= pd.to_datetime("09:15").time() and t < pd.to_datetime("10:00").time():
        return "OPEN"
    elif t >= pd.to_datetime("10:00").time() and t < pd.to_datetime("13:00").time():
        return "MIDDAY"
    else:
        return "CLOSE"

aarti["block"] = aarti["time"].apply(time_block)

aarti[["timestamp", "block", "volume"]].head(250)


In [ ]:
aarti_block_summary = (
    aarti
    .groupby(["date", "block"])
    .agg(
        total_minutes=("timestamp", "count"),
        traded_minutes=("volume", lambda x: (x > 0).sum()),
        zero_minutes=("volume", lambda x: (x == 0).sum())
    )
    .reset_index()
)

aarti_block_summary["dead_fraction"] = (
    aarti_block_summary["zero_minutes"] / aarti_block_summary["total_minutes"]
)

aarti_block_summary.head(12)


In [ ]:
import os
import pandas as pd


# BASE_DIR = os.path.dirname(os.getcwd())  # project root (one level above notebooks)
# PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "clean_intraday")

print("Cleaned intraday folder:", PROCESSED_DIR)
print("Example files:", os.listdir(PROCESSED_DIR)[:5])


In [9]:
def summarize_one_stock_blocks(clean_path: str) -> pd.DataFrame:
    """
    Given a CLEAN intraday CSV (one stock),
    return a per-day, per-block summary with dead_fraction.
    Blocks:
      - OPEN   : 09:15–10:00
      - MIDDAY : 10:00–13:00
      - CLOSE  : 13:00–15:30
    """
    df = pd.read_csv(clean_path, parse_dates=["timestamp"])
    
    # Symbol from filename
    filename = os.path.basename(clean_path)
    symbol = filename.split("__")[0]
    
    # Date and time
    df["date"] = df["timestamp"].dt.date
    df["time"] = df["timestamp"].dt.time

    # Time → block mapping
    def time_block(t):
        t_open_start  = pd.to_datetime("09:15").time()
        t_open_end    = pd.to_datetime("10:00").time()
        t_mid_start   = pd.to_datetime("10:00").time()
        t_mid_end     = pd.to_datetime("13:00").time()
        # rest goes to CLOSE

        if t >= t_open_start and t < t_open_end:
            return "OPEN"
        elif t >= t_mid_start and t < t_mid_end:
            return "MIDDAY"
        else:
            return "CLOSE"

    df["block"] = df["time"].apply(time_block)

    # Group by date + block
    summary = (
        df.groupby(["date", "block"])
          .agg(
              total_minutes=("timestamp", "count"),
              traded_minutes=("volume", lambda x: (x > 0).sum()),
              zero_minutes=("volume",  lambda x: (x == 0).sum())
          )
          .reset_index()
    )

    summary["dead_fraction"] = summary["zero_minutes"] / summary["total_minutes"]
    summary["symbol"] = symbol

    # Reorder columns for clarity
    summary = summary[["symbol", "date", "block",
                       "total_minutes", "traded_minutes",
                       "zero_minutes", "dead_fraction"]]
    return summary


In [18]:
# List all cleaned CSVs again
clean_files = [
    os.path.join(PROCESSED_DIR, f)
    for f in os.listdir(PROCESSED_DIR)
    if f.endswith(".csv")
]

len(clean_files), clean_files[:5]


(150,
 ['C:\\data\\processed\\clean_intraday\\AARTIIND__CLEAN__NSE__MINUTE.csv',
  'C:\\data\\processed\\clean_intraday\\ABCAPITAL__CLEAN__NSE__MINUTE.csv',
  'C:\\data\\processed\\clean_intraday\\ABFRL__CLEAN__NSE__MINUTE.csv',
  'C:\\data\\processed\\clean_intraday\\ADANIENT__CLEAN__NSE__MINUTE.csv',
  'C:\\data\\processed\\clean_intraday\\ADANIGAS__CLEAN__NSE__MINUTE.csv'])

In [20]:
print("Kernel alive after interrupt")


Kernel alive after interrupt


In [ ]:
# List all cleaned CSVs
clean_files = [
    os.path.join(PROCESSED_DIR, f)
    for f in os.listdir(PROCESSED_DIR)
    if f.endswith(".csv")
]
print("Number of cleaned files:", len(clean_files))

# Output path for the block-level summary
BLOCKS_OUT_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(BLOCKS_OUT_DIR, exist_ok=True)

blocks_path = os.path.join(BLOCKS_OUT_DIR, "intraday_block_liquidity.csv")

# If file exists from a previous attempt, remove it so we start fresh
if os.path.exists(blocks_path):
    os.remove(blocks_path)

print("Writing block-level liquidity summary to:", blocks_path)

files_to_process = clean_files  # all stocks; for testing you could use clean_files[:10]

for i, path in enumerate(files_to_process):
    symbol_file = os.path.basename(path)
    print(f"[{i+1}/{len(files_to_process)}] {symbol_file} ...")

    try:
        summary = summarize_one_stock_blocks(path)
    except Exception as e:
        print("  Error:", e)
        continue

    # Append to CSV on disk
    mode = "w" if i == 0 else "a"       # first file → write, rest → append
    header = (i == 0)                   # header only for first write

    summary.to_csv(blocks_path, mode=mode, header=header, index=False)


Number of cleaned files: 150
Writing block-level liquidity summary to: C:\data\processed\intraday_block_liquidity.csv
[1/150] AARTIIND__CLEAN__NSE__MINUTE.csv ...


In [ ]:
blocks_partial = pd.read_csv(blocks_path)
print("Current rows in block file:", len(blocks_partial))
blocks_partial.head()


In [ ]:
processed_symbols = set(blocks_partial["symbol"].unique())
len(processed_symbols), list(processed_symbols)[:10]


In [ ]:
clean_files = [
    os.path.join(PROCESSED_DIR, f)
    for f in os.listdir(PROCESSED_DIR)
    if f.endswith(".csv")
]

all_symbols = [os.path.basename(f).split("__")[0] for f in clean_files]
len(all_symbols)


In [ ]:
remaining_files = [
    f for f in clean_files
    if os.path.basename(f).split("__")[0] not in processed_symbols
]

print("Remaining stocks to process:", len(remaining_files))
[os.path.basename(f) for f in remaining_files[:10]]


In [ ]:
import numpy as np

def summarize_one_stock_blocks(clean_path: str) -> pd.DataFrame:
    df = pd.read_csv(clean_path, parse_dates=["timestamp"])
    
    filename = os.path.basename(clean_path)
    symbol = filename.split("__")[0]
    
    df["date"] = df["timestamp"].dt.date

    mins = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute

    open_start  = 9 * 60 + 15
    open_end    = 10 * 60
    mid_start   = 10 * 60
    mid_end     = 13 * 60

    conds = [
        (mins >= open_start) & (mins < open_end),
        (mins >= mid_start)  & (mins < mid_end)
    ]
    choices = ["OPEN", "MIDDAY"]
    df["block"] = np.select(conds, choices, default="CLOSE")

    summary = (
        df.groupby(["date", "block"])
          .agg(
              total_minutes=("timestamp", "count"),
              traded_minutes=("volume", lambda x: (x > 0).sum()),
              zero_minutes=("volume",  lambda x: (x == 0).sum())
          )
          .reset_index()
    )

    summary["dead_fraction"] = summary["zero_minutes"] / summary["total_minutes"]
    summary["symbol"] = symbol

    summary = summary[["symbol", "date", "block",
                       "total_minutes", "traded_minutes",
                       "zero_minutes", "dead_fraction"]]
    return summary


In [ ]:
for i, path in enumerate(remaining_files):
    symbol_file = os.path.basename(path)
    print(f"[RESUME {i+1}/{len(remaining_files)}] {symbol_file}")

    try:
        summary = summarize_one_stock_blocks(path)
    except Exception as e:
        print("  Error:", e)
        continue

    summary.to_csv(blocks_path, mode="a", header=False, index=False)


In [5]:
# Make sure the parent directory for the block file exists
blocks_dir = os.path.dirname(blocks_path)
os.makedirs(blocks_dir, exist_ok=True)

print("Block directory confirmed:", blocks_dir)
print("Writing block-level liquidity summary to:", blocks_path)


Block directory confirmed: C:\Indian_microstructure\data\processed
Writing block-level liquidity summary to: C:\Indian_microstructure\data\processed\intraday_block_liquidity.csv


In [6]:

# Rebuild full list of cleaned files (150)
clean_files = [
    os.path.join(PROCESSED_DIR, f)
    for f in os.listdir(PROCESSED_DIR)
    if f.endswith(".csv")
]
print("Number of cleaned files:", len(clean_files))

# If an old block file exists, delete it so we start clean
if os.path.exists(blocks_path):
    os.remove(blocks_path)
    print("Old block file removed.")

print("Writing block-level liquidity summary to:", blocks_path)

for i, path in enumerate(clean_files):
    symbol_file = os.path.basename(path)
    print(f"[{i+1}/{len(clean_files)}] {symbol_file}")

    try:
        summary = summarize_one_stock_blocks(path)
    except Exception as e:
        print("  Error:", e)
        continue

    mode = "w" if i == 0 else "a"
    header = (i == 0)

    summary.to_csv(blocks_path, mode=mode, header=header, index=False)


Number of cleaned files: 150
Writing block-level liquidity summary to: C:\Indian_microstructure\data\processed\intraday_block_liquidity.csv
[1/150] AARTIIND__CLEAN__NSE__MINUTE.csv
[2/150] ABCAPITAL__CLEAN__NSE__MINUTE.csv
[3/150] ABFRL__CLEAN__NSE__MINUTE.csv
[4/150] ADANIENT__CLEAN__NSE__MINUTE.csv
[5/150] ADANIGAS__CLEAN__NSE__MINUTE.csv
[6/150] ADANIPORTS__CLEAN__NSE__MINUTE.csv
[7/150] AJANTPHARM__CLEAN__NSE__MINUTE.csv
[8/150] AMARAJABAT__CLEAN__NSE__MINUTE.csv
[9/150] APLLTD__CLEAN__NSE__MINUTE.csv
[10/150] APOLLOHOSP__CLEAN__NSE__MINUTE.csv
[11/150] APOLLOTYRE__CLEAN__NSE__MINUTE.csv
[12/150] ASHOKLEY__CLEAN__NSE__MINUTE.csv
[13/150] ASIANPAINT__CLEAN__NSE__MINUTE.csv
[14/150] AUBANK__CLEAN__NSE__MINUTE.csv
[15/150] AXISBANK__CLEAN__NSE__MINUTE.csv
[16/150] BAJAJFINSV__CLEAN__NSE__MINUTE.csv
[17/150] BAJAJ_AUTO__CLEAN__NSE__MINUTE.csv
[18/150] BAJFINANCE__CLEAN__NSE__MINUTE.csv
[19/150] BALKRISIND__CLEAN__NSE__MINUTE.csv
[20/150] BANKINDIA__CLEAN__NSE__MINUTE.csv
[21/150] BATAI

In [7]:
blocks_df = pd.read_csv(blocks_path)
print("Final block table shape:", blocks_df.shape)
print("Unique symbols:", blocks_df["symbol"].nunique())
blocks_df.head()


Final block table shape: (433914, 7)
Unique symbols: 150


,symbol,date,block,total_minutes,traded_minutes,zero_minutes,dead_fraction
0,AARTIIND,2017-01-02,CLOSE,150,57,93,0.620000
1,AARTIIND,2017-01-02,MIDDAY,180,46,134,0.744444
2,AARTIIND,2017-01-02,OPEN,45,27,18,0.400000
3,AARTIIND,2017-01-03,CLOSE,150,97,53,0.353333
4,AARTIIND,2017-01-03,MIDDAY,180,86,94,0.522222
